# IADD YOLOv11 - Run 5 finish (low-LR fine-tune)

Continues from run5's best.pt for ~10 epochs with mosaic + mixup OFF and a low
learning rate. This is the "close_mosaic" finishing step that never triggered during
run5 (the frequent account-switch resumes reset it): the model gets polished on
realistic, non-composite images, which typically adds ~0.5-1.5 mAP.

Output run name = run5_ft (run5 stays untouched as the fallback).

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Install Ultralytics

In [ ]:
!pip install -q ultralytics
import ultralytics; ultralytics.checks()

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Unzip the dataset into Colab

In [ ]:
!rm -rf /content/iadd_subset_v4
!unzip -q "/content/drive/MyDrive/iadd_subset_v4.zip" -d /content
!ls /content/iadd_subset_v4

## 5. Finish fine-tune - from run5 best.pt, or auto-resume

10 epochs, mosaic+mixup off, low LR. optimizer/lr are set explicitly (not 'auto')
so the low learning rate is actually used, not overridden.

In [ ]:
from ultralytics import YOLO
import os

RUNS     = '/content/drive/MyDrive/iadd_runs'
DATA     = '/content/iadd_subset_v4/data.yaml'
RUN_NAME = 'run5_ft'
BASE     = f'{RUNS}/run5/weights/best.pt'
CKPT     = f'{RUNS}/{RUN_NAME}/weights/last.pt'

if os.path.exists(CKPT):
    print('resuming finish ->', CKPT)
    model = YOLO(CKPT)
    model.train(resume=True)
else:
    print('starting 10-epoch finish from run5 best.pt')
    model = YOLO(BASE)
    model.train(
        data=DATA, epochs=10, imgsz=1280, batch=12,
        project=RUNS, name=RUN_NAME, exist_ok=True, plots=True,
        optimizer='SGD', lr0=0.001, lrf=0.01, warmup_epochs=0.0,  # gentle low-LR finish
        mosaic=0.0, mixup=0.0,        # realistic single images only (the close_mosaic effect)
        hsv_v=0.4, degrees=5.0,       # keep mild, realistic single-image augmentation
    )

## 6. Validate - honest metrics + operating confidence + clean matrix

In [ ]:
from ultralytics import YOLO
import numpy as np, os, shutil

DATA = '/content/iadd_subset_v4/data.yaml'
RUNS = '/content/drive/MyDrive/iadd_runs'
best = YOLO(f'{RUNS}/run5_ft/weights/best.pt')

mv = best.val(data=DATA, split='val', plots=True, project=RUNS, name='run5ft_val', exist_ok=True)
print('VAL  mAP50=%.4f  mAP50-95=%.4f' % (mv.box.map50, mv.box.map))

CONF = 0.25
for x, y, _, ylabel in mv.curves_results:
    if ylabel == 'F1':
        px = np.asarray(x); y = np.asarray(y)
        f1 = y.mean(0) if y.ndim > 1 else y
        CONF = round(float(px[f1.argmax()]), 3)
print('operating confidence (from val F1 curve) =', CONF)

t = best.val(data=DATA, split='val', conf=CONF, plots=True, project=RUNS, name='run5ft_val_tmp', exist_ok=True)
for f in ('confusion_matrix.png', 'confusion_matrix_normalized.png'):
    shutil.copy(f'{RUNS}/run5ft_val_tmp/{f}', f'{RUNS}/run5ft_val/{f}')
shutil.rmtree(f'{RUNS}/run5ft_val_tmp')
print('run5ft_val ready (conf %.3f)' % CONF)

## 7. Test - honest metrics + clean matrix at the same confidence

In [ ]:
mt = best.val(data=DATA, split='test', plots=True, project=RUNS, name='run5ft_test', exist_ok=True)
print('TEST mAP50=%.4f  mAP50-95=%.4f' % (mt.box.map50, mt.box.map))

t = best.val(data=DATA, split='test', conf=CONF, plots=True, project=RUNS, name='run5ft_test_tmp', exist_ok=True)
for f in ('confusion_matrix.png', 'confusion_matrix_normalized.png'):
    shutil.copy(f'{RUNS}/run5ft_test_tmp/{f}', f'{RUNS}/run5ft_test/{f}')
shutil.rmtree(f'{RUNS}/run5ft_test_tmp')
print('run5ft_test ready (conf %.3f)' % CONF)

## 8. Pooled val+test evaluation - the robust headline number

val and test are disjoint unseen-video subsets that differ in difficulty. Evaluating
on the two POOLED (a temporary combined split) averages out that variance and gives a
single robust mAP over all ~83 unseen videos. It is also conservative, because the
harder val is included.

In [ ]:
import glob, os, yaml

# build a temporary pooled split = val + test (symlinks, no copying)
POOL = '/content/iadd_pool'
os.system(f'rm -rf {POOL}')
os.makedirs(f'{POOL}/images/pool', exist_ok=True)
os.makedirs(f'{POOL}/labels/pool', exist_ok=True)
for sp in ('val', 'test'):
    for img in glob.glob(f'/content/iadd_subset_v4/images/{sp}/*.jpg'):
        b = os.path.basename(img)
        os.symlink(img, f'{POOL}/images/pool/{b}')
        lb = f'/content/iadd_subset_v4/labels/{sp}/' + b[:-4] + '.txt'
        if os.path.exists(lb):
            os.symlink(lb, f'{POOL}/labels/pool/' + b[:-4] + '.txt')
with open(f'{POOL}/data.yaml', 'w') as f:
    yaml.safe_dump({'path': POOL, 'val': 'images/pool', 'nc': 6,
                    'names': ['person','car','motorcycle','bus','truck','traffic_light']}, f)

mp = best.val(data=f'{POOL}/data.yaml', split='val', plots=False,
              project=RUNS, name='run5ft_pool', exist_ok=True)
print('POOLED (val+test)  mAP50=%.4f  mAP50-95=%.4f  images=%d'
      % (mp.box.map50, mp.box.map, len(glob.glob(f'{POOL}/images/pool/*.jpg'))))

---
Results on Drive under `iadd_runs/`:
- `run5_ft/` - the finished weights
- `run5ft_val/`, `run5ft_test/` - honest curves + clean confusion matrix
- `run5ft_pool/` - pooled val+test evaluation (robust headline mAP)